In [1]:
!pip install streamlit transformers torch pyngrok -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 50.8 MB/s eta 0:00:00


In [13]:
%%writefile app.py
import streamlit as st
from transformers import AutoTokenizer,AutoModelForCausalLM
import torch

## Page setup
st.set_page_config(page_title="GPT ChatBot",page_icon="🤖")
st.title("GPT ChatBot (Instruct Model)")
st.caption("Pure decoder-only GPT in action: each reply is generated from your entire conversation history")

## Load Model (With caching)
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

@st.cache_resource
def load_model():
  tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
  model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
  return tokenizer,model
tokenizer,model = load_model()

## Session State Setup

if "messages" not in st.session_state:
  st.session_state.messages = []

## Show Old messages
for msg in st.session_state.messages:
  with st.chat_message(msg["role"]):
    st.write(msg["content"])

## Input Message
user_input = st.chat_input("Message Type...")

if user_input:
  with st.chat_message("user"):
    st.write(user_input)
  st.session_state.messages.append({"role":"user","content":user_input})

  ## Tokenize and concatenate with history
  SYSTEM_PROMPT = {"role": "system", "content": "You are a helpful, concise assistant."}
  new_input_ids = tokenizer.apply_chat_template(
        [SYSTEM_PROMPT] + st.session_state.messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )



  with st.chat_message("assistant"):
    with st.spinner("Typing"):
      ## Response Generation
      output_token_ids=model.generate(
      **new_input_ids,
      max_new_tokens=150,
      pad_token_id=tokenizer.eos_token_id,
      do_sample=True,
      temperature=0.7,
      top_k=50,
      top_p=0.9,
      )

      ## New Reply
      response=tokenizer.decode(
      output_token_ids[0][new_input_ids["input_ids"].shape[-1]:],skip_special_tokens=True
      )

    st.write(response)
  st.session_state.messages.append({"role":"assistant","content":response})

## Reset Button
if st.button("Reset Chat"):
  st.session_state.messages = []
  st.rerun()

Overwriting app.py


### **Colab AuthToken**
---

In [9]:
from pyngrok import ngrok
ngrok.set_auth_token("3FOKb1bJlDWtpZKBdjoQyokOXjC_82xUw4o7NxgztXqZqVv7o")

### **Run App + Open Tunnel**
---

In [14]:
import subprocess
process = subprocess.Popen(["streamlit","run","app.py","--server.port","8501"])

public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://acutely-protract-refried.ngrok-free.dev" -> "http://localhost:8501"


In [12]:
ngrok.kill()

In [ ]:
process.terminate()